# Матрица компетенций курса ДПО

**Тема главы:** матрица компетенций курса ДПО «ИИ-инструменты в преподавательской, научной и повседневной работе: от понимания принципов до практического применения».

**Область главы:** компетентностная модель курса: ПК-ИИ-1 — ПК-ИИ-10.

**Назначение ноутбука:** показать, как методическая модель курса превращается в воспроизводимые данные, таблицы, графы связей, жизненный цикл подтверждения компетенции и проверку полноты матрицы.

Ноутбук подготовлен как публичная глава Jupyter Book-книги проекта. Он не создает PDF и Word-файлы, не использует ручной HTML и не обращается к закрытым внешним сервисам.


## Постановка задачи

Необходимо разработать строго оформленную матрицу компетенций для программы повышения квалификации объемом 72 часа. Матрица должна связывать:

- 10 профессиональных компетенций ПК-ИИ-1 — ПК-ИИ-10;
- 8 модулей курса;
- знания, умения и профессиональные действия слушателя;
- проверяемые артефакты;
- тесты или практические проверки;
- критерии освоения;
- уровни сформированности;
- типичные ошибки и риски;
- итоговый проект и документ о прохождении обучения.

Главный принцип: каждая компетенция должна быть проверяема не общей формулировкой, а конкретным профессиональным действием преподавателя.

## Контекст и границы главы

Глава не описывает всю образовательную платформу и не раскрывает содержание всех лекций. Ее задача — зафиксировать методическую основу компетентностной модели, на которой дальше строятся:

- рабочая программа модуля;
- фонд оценочных средств;
- итоговый проект;
- структура курса в образовательной платформе;
- сертификат или иной документ об обучении;
- научные статьи о методике формирования ИИ-компетенций преподавателя.

Правовые и нормативные положения в этой главе не формулируются как окончательные юридические требования. Если требуется официальная формулировка, она должна проверяться по внутренним документам образовательной организации.


## Структура ноутбука и ожидаемые результаты

| № | Тип | Назначение ячейки | Ожидаемый результат |
|---:|---|---|---|
| 1 | Markdown | Заголовок главы | Вводный блок для Jupyter Book |
| 2 | Markdown | Постановка задачи | Граница компетентностной модели |
| 3 | Markdown | Контекст и ограничения | Объяснение методической роли матрицы |
| 4 | Code | Импорт библиотек и настройка путей | Подготовлена среда анализа |
| 5 | Code | Загрузка структурированных данных | Сформирован словарь `competency_matrix` |
| 6 | Code | Проверка и сохранение данных | Созданы локальные YAML и JSON при выполнении ноутбука |
| 7 | Code | Таблица модулей | DataFrame с 8 модулями |
| 8 | Code | Таблица компетенций | DataFrame с 10 компетенциями |
| 9 | Code | Матрица «компетенции — модули» | Табличная матрица связей |
| 10 | Code | Диаграмма распределения часов | График часов по модулям |
| 11 | Code | Тепловая карта связей | Схема покрытия модулей компетенциями |
| 12 | Code | Граф связей модулей и компетенций | NetworkX-граф методической структуры |
| 13 | Code | Жизненный цикл подтверждения компетенции | Ориентированный граф процесса |
| 14 | Code | Алгоритм проверки полноты | Таблица проблем и полноты заполнения |
| 15 | Code | Пример данных по слушателю | Демонстрация итогового статуса компетенций |
| 16 | Markdown | Интерпретация результатов | Выводы по структуре матрицы |


In [ ]:
from pathlib import Path
import json
import yaml
from dataclasses import dataclass, asdict

import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "figures"
DATA_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_colwidth", 120)

print("Рабочий каталог:", BASE_DIR)
print("Каталог данных:", DATA_DIR)
print("Каталог схем:", FIGURES_DIR)

In [ ]:
data_candidates = [
    BASE_DIR / "data" / "branch_02_competency_matrix.json",
    BASE_DIR / "book" / "02_dpo_course_design" / "data" / "branch_02_competency_matrix.json",
    BASE_DIR / "data" / "branches" / "02_competency_matrix" / "branch_02_competency_matrix.json",
]

for candidate in data_candidates:
    if candidate.exists():
        source_path = candidate
        break
else:
    raise FileNotFoundError("Не найден файл branch_02_competency_matrix.json")

competency_matrix = json.loads(source_path.read_text(encoding="utf-8"))
print("Загружена матрица компетенций:", source_path)
print("Количество компетенций:", len(competency_matrix["competencies"]))


In [ ]:
json_output = DATA_DIR / "branch_02_competency_matrix.json"
yaml_output = DATA_DIR / "branch_02_competency_matrix.yaml"

json_output.write_text(
    json.dumps(competency_matrix, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

yaml_output.write_text(
    yaml.safe_dump(competency_matrix, allow_unicode=True, sort_keys=False, width=100),
    encoding="utf-8"
)

print("Сохранено:")
print("-", json_output)
print("-", yaml_output)

## Данные курса: модули и трудоемкость

Первый уровень анализа — проверка базовой структуры курса. Для матрицы компетенций важно, чтобы каждая компетенция была привязана к конкретным модулям, а не существовала отдельно от учебной траектории.

In [ ]:
modules_df = pd.DataFrame(competency_matrix["course"]["modules"])
modules_df = modules_df[["id", "number", "title", "hours"]]
modules_df

## Данные компетенций

Следующая таблица является человекочитаемой проекцией структурированных данных. Она не заменяет YAML/JSON, но позволяет быстро увидеть общий контур компетентностной модели.

In [ ]:
competencies = competency_matrix["competencies"]
competencies_df = pd.DataFrame([
    {
        "code": c["code"],
        "name": c["name"],
        "modules": ", ".join(c["formation_modules"]),
        "artifact": c["artifact"]["title"],
        "check_type": c["test_or_check"]["type"],
        "criteria_count": len(c["assessment_criteria"]),
        "errors_count": len(c["typical_errors"]),
        "risks_count": len(c["risks"]),
    }
    for c in competencies
])
competencies_df

## Матрица связей «компетенции — модули»

Матрица показывает, в каких модулях формируется каждая компетенция. Такое представление полезно для проверки баланса курса: компетенции не должны быть оторваны от учебного плана, а модули не должны быть содержательно пустыми.

In [ ]:
module_ids = [m["id"] for m in competency_matrix["course"]["modules"]]
rows = []
for c in competencies:
    row = {"competency": c["code"], "name": c["name"]}
    for mid in module_ids:
        row[mid] = 1 if mid in c["formation_modules"] else 0
    rows.append(row)

coverage_df = pd.DataFrame(rows)
coverage_df

## Диаграмма распределения часов по модулям

Распределение часов используется как контрольная рамка: оно показывает, где сосредоточена основная практическая нагрузка и где логично располагать ключевые артефакты.

In [ ]:
ax = modules_df.plot(kind="bar", x="id", y="hours", legend=False, figsize=(9, 4))
ax.set_title("Распределение часов по модулям курса")
ax.set_xlabel("Модуль")
ax.set_ylabel("Часы")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Тепловая карта покрытия компетенций модулями

Тепловая карта визуализирует, какие модули участвуют в формировании каждой компетенции. Она помогает обнаружить методические перекосы: например, компетенцию без модулей формирования или модуль без компетентностной нагрузки.

In [ ]:
heatmap_data = coverage_df[module_ids].values
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(heatmap_data, aspect="auto")
ax.set_xticks(range(len(module_ids)))
ax.set_xticklabels(module_ids)
ax.set_yticks(range(len(coverage_df)))
ax.set_yticklabels(coverage_df["competency"])
ax.set_title("Покрытие компетенций модулями")
ax.set_xlabel("Модули")
ax.set_ylabel("Компетенции")

for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        if heatmap_data[i, j] == 1:
            ax.text(j, i, "•", ha="center", va="center", fontsize=14)

plt.tight_layout()
plt.show()

## Граф связей модулей, компетенций и итогового проекта

Граф показывает, что матрица компетенций является не перечнем формулировок, а системой связей. Компетенции формируются в модулях, подтверждаются артефактами и сходятся в итоговом проекте.

In [ ]:
G = nx.Graph()

for m in competency_matrix["course"]["modules"]:
    G.add_node(m["id"], label=m["title"], kind="module")

for c in competencies:
    G.add_node(c["code"], label=c["name"], kind="competency")
    for mid in c["formation_modules"]:
        G.add_edge(mid, c["code"], relation="forms")

G.add_node("FINAL_PROJECT", label="Итоговый проект", kind="final_project")
for c in competencies:
    G.add_edge(c["code"], "FINAL_PROJECT", relation="integrates")

pos = nx.spring_layout(G, seed=42, k=0.8)
fig, ax = plt.subplots(figsize=(12, 8))

module_nodes = [n for n, d in G.nodes(data=True) if d["kind"] == "module"]
competency_nodes = [n for n, d in G.nodes(data=True) if d["kind"] == "competency"]
final_nodes = ["FINAL_PROJECT"]

nx.draw_networkx_nodes(G, pos, nodelist=module_nodes, node_size=950, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=competency_nodes, node_size=850, ax=ax)
nx.draw_networkx_nodes(G, pos, nodelist=final_nodes, node_size=1400, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.35, ax=ax)
nx.draw_networkx_labels(G, pos, labels={n: n for n in G.nodes()}, font_size=9, ax=ax)

ax.set_title("Граф связей: модули — компетенции — итоговый проект")
ax.axis("off")
plt.tight_layout()
plt.show()

## Диаграмма жизненного цикла подтверждения компетенции

Одна компетенция считается освоенной только после двухчастного подтверждения: принятого артефакта и пройденного теста или практической проверки. Для итоговых и спорных результатов решение должно подтверждаться человеком.

In [ ]:
lifecycle = nx.DiGraph()
stages = [
    "Формулировка компетенции",
    "Модуль курса",
    "Учебное действие",
    "Артефакт",
    "Тест / практическая проверка",
    "Проверка по критериям",
    "Статус компетенции",
    "Итоговый проект",
    "Сертификат / документ",
]
for stage in stages:
    lifecycle.add_node(stage)
for a, b in zip(stages, stages[1:]):
    lifecycle.add_edge(a, b)

pos = {stage: (index, 0) for index, stage in enumerate(stages)}
fig, ax = plt.subplots(figsize=(13, 4))
nx.draw_networkx_nodes(lifecycle, pos, node_size=1700, ax=ax)
nx.draw_networkx_edges(lifecycle, pos, arrows=True, arrowstyle="-|>", arrowsize=18, ax=ax)
nx.draw_networkx_labels(lifecycle, pos, font_size=8, ax=ax)
ax.set_title("Жизненный цикл подтверждения компетенции")
ax.axis("off")
plt.tight_layout()
plt.show()

## Алгоритм проверки полноты матрицы

Следующий фрагмент демонстрирует воспроизводимую проверку качества данных. Такая проверка нужна для методической и публикационной надежности: она позволяет обнаружить неполные компетенции до включения матрицы в РПМ, ФОС или цифровую платформу.


In [ ]:
@dataclass
class ValidationIssue:
    code: str
    field: str
    problem: str

REQUIRED_FIELDS = [
    "code",
    "name",
    "formulation",
    "formation_modules",
    "must_know",
    "must_be_able_to",
    "professional_actions",
    "artifact",
    "test_or_check",
    "assessment_criteria",
    "levels",
    "typical_errors",
    "risks",
    "final_project_connection",
]

issues = []
for c in competencies:
    for field in REQUIRED_FIELDS:
        if field not in c or c[field] in (None, "", [], {}):
            issues.append(ValidationIssue(c.get("code", "UNKNOWN"), field, "field is empty or missing"))
    for mid in c.get("formation_modules", []):
        if mid not in module_ids:
            issues.append(ValidationIssue(c["code"], "formation_modules", f"unknown module id: {mid}"))
    if len(c.get("levels", {})) < 4:
        issues.append(ValidationIssue(c["code"], "levels", "expected at least 4 levels: 0, 1, 2, 3"))

validation_df = pd.DataFrame([asdict(issue) for issue in issues])
if validation_df.empty:
    print("Проверка пройдена: критических пропусков в матрице не обнаружено.")
else:
    display(validation_df)

## Демонстрация расчета статуса компетенций для условного слушателя

Этот пример не является официальной системой оценивания. Он показывает, как двухчастная модель может быть представлена в данных: компетенция засчитывается только тогда, когда принят артефакт и пройдена проверка.

In [ ]:
sample_records = []
for i, c in enumerate(competencies):
    artifact_status = "accepted" if i not in (4, 8) else "needs_revision"
    test_status = "passed" if i not in (8,) else "not_passed"
    competency_status = "mastered" if artifact_status == "accepted" and test_status == "passed" else "partial"
    sample_records.append({
        "learner_id": "demo_learner_001",
        "competency": c["code"],
        "artifact_status": artifact_status,
        "test_status": test_status,
        "competency_status": competency_status,
    })

sample_status_df = pd.DataFrame(sample_records)
summary = sample_status_df["competency_status"].value_counts().rename_axis("status").reset_index(name="count")

display(sample_status_df)
display(summary)

if (sample_status_df["competency_status"] == "mastered").all():
    print("Итог: все компетенции освоены, возможна подготовка документа об обучении.")
else:
    missing = sample_status_df.loc[sample_status_df["competency_status"] != "mastered", "competency"].tolist()
    print("Итог: требуется доработка компетенций:", ", ".join(missing))

## Интерпретация результатов

Матрица компетенций выполняет четыре функции.

1. **Методическая функция:** переводит общую цель курса в проверяемые профессиональные действия преподавателя.
2. **Оценочная функция:** задает двухчастное подтверждение компетенции через артефакт и тест или практическую проверку.
3. **Платформенная функция:** дает структуру для сущностей курса, заданий, проверок, прогресса и сертификата.
4. **Исследовательская функция:** позволяет анализировать, какие компетенции формируются в каких модулях, какие артефакты подтверждают освоение и какие риски мешают достижению результата.

Ключевой вывод: компетенция не считается освоенной по факту просмотра лекции. Освоение подтверждается только через действие слушателя, проверяемый продукт и формализованную проверку.

## Публикационное использование ноутбука

В книге этот ноутбук выполняет роль демонстрационной и проверяемой главы: он показывает не только список компетенций, но и связи между модулями, артефактами, проверками и итоговым проектом.

Если структура книги будет меняться, файл должен оставаться в разделе курса ДПО и быть подключен через `_toc.yml` как воспроизводимая глава матрицы компетенций.


## Дальнейшее развитие

Материалы главы могут быть использованы при подготовке отдельной статьи о компетентностной модели курса ДПО по ИИ-инструментам. В текущей публичной версии фиксируются только те элементы, которые уже развернуты в книге: десять компетенций ПК-ИИ-1 — ПК-ИИ-10, связь с модулями, артефактами, проверками, итоговым проектом и цифровым сопровождением образовательного результата.

Дополнительные исследовательские направления следует развивать отдельными публикационными задачами, чтобы не перегружать демонстрационную главу неподготовленными тезисами.
